# Logits Preprocessing and Data Engineering

In [1]:
def default_params(): 
    return {
        'current_model': 'M2', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'name': '/workspaces/CodeSmells/semeru-datasets/code_smells/codesmell_dataset.csv',
            'content_column': 'code',
        },
        'default_max_position_embeddings' : 16384,
        'output_path': '../data/raw_logits',
        'preprocessed_dataset_dir' : '../datax/code_smells/dataset_preprocessing',
        'cache_dir': '../datax/hugging_face_cache',
        'log_file': '../datax/code_smells/logit_extraction.log', 
        'callbacks_dir' : '../datax/code_smells/callbacks',
        'causal_models': {
            'M2': 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/codellama/CodeLlama-13b-hf
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
from transformers import AutoTokenizer, MistralForCausalLM 
from datasets import load_dataset

In [4]:
import logging
#logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)
logging.basicConfig(
    filename=params['log_file'],
    filemode='a',
    format='%(asctime)s : %(levelname)s : %(message)s', 
    level=logging.INFO
    )

In [5]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

#### Dataset

In [17]:
df_dataset = pd.read_json(params['preprocessed_dataset_dir'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '.json', )

In [ ]:
df_dataset.reset_index(drop=True, inplace=True)

#### Model Loading

In [6]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded LlamaTokenizer - " + model_name)
     model = None
     match params['quantization']:
               case 'int4':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
               case 'int8':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
               case 'float32':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
               case 'float16':
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
               case _: 
                    model = MistralForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded MistralForCausalLM - " + model_name)

     return tokenizer, model

In [7]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

#### Softmax Normalization and Data Engineering

In [8]:
def topk_tuple( logit_vocab_tensor, largest, tokenizer_fn):
    "Run topk for a token"
    topk = logit_vocab_tensor.topk( k=1 , largest=largest ) #TODO K number of elements can be extended
    return ( tokenizer.convert_tokens_to_string([tokenizer_fn.decode(topk.indices)]), topk.values.item())

def min_max_logits( logit_vocab_sample_tensor, tokenizer_fn ):
    "Compute min_max for a sample"
    max_cases = []
    min_cases = []
    for logit_vocab_tensor in logit_vocab_sample_tensor:
        max_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = True, tokenizer_fn = tokenizer_fn) ) #TST Max Logit
        min_cases.append( topk_tuple( logit_vocab_tensor = logit_vocab_tensor, largest = False, tokenizer_fn = tokenizer_fn) ) #TST Min Logit
    return max_cases, min_cases

def actual_logit( 
                 logit_vocab_sample_tensor, 
                 tokenized_prompt, 
                 tokenizer_fn,
                 ):
    "Compute actual logits for a sample"
    actual_logits_prompt = []
    for token_pos, id_token in enumerate( tokenized_prompt[1:] ): #Eliminate the first token prediction since we do not use it
        actual_logits_prompt.append(
            (   tokenizer.convert_tokens_to_string([tokenizer_fn.decode( int(id_token))]), #retrieving the name of the token with the id
                logit_vocab_sample_tensor[token_pos][int(id_token)].item()) #retrieving the logit given the position in the sequence and the position in the vocab
            )
    return actual_logits_prompt

In [9]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [18]:
out= np.load(params['callbacks_dir']+ '/'+ params['current_model'] + '_q_' + params['quantization'] +'/' + 'logits_tensor[0]_batch[0].npy')
print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 457, 32768)


In [19]:
max_case,min_case = min_max_logits(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out], ####### 
    tokenizer_fn= tokenizer
    )
print(max_case)
assert len(max_case) == len(min_case)

[('#', 0.2665287256240845), ('solution', 0.058843858540058136), ('_', 0.7111697196960449), ('get', 0.037967000156641006), ('_', 0.028935091570019722), ('_', 0.43795230984687805), ('self', 0.142356738448143), ('):', 0.7956698536872864), ('\n', 0.9903011322021484), (' ', 0.564018726348877), ('"""', 0.16692335903644562), ('=', 0.8424184322357178), ('{', 0.3088832497596741), ('\n', 0.7759588360786438), ('         ', 0.8008356690406799), ("'", 0.332145094871521), ('a', 0.2342398315668106), ("':", 0.9017483592033386), ("{'", 0.3621208071708679), ('a', 0.7407320141792297), ("':", 0.8090277314186096), ("'", 0.6437082290649414), ('1', 0.26039478182792664), ("',", 0.7275657057762146), ("'", 0.9586458802223206), ('2', 0.6101283431053162), ("':", 0.996057391166687), ("'", 0.9527722597122192), ('b', 0.6182431578636169), ("'},", 0.8787501454353333), ('\n', 0.9783669114112854), ('         ', 0.9917137622833252), ("'", 0.9982050657272339), ('b', 0.9849040508270264), ("':", 0.9993496537208557), ("{'", 

In [20]:
assert tokenizer.decode(df_dataset['input_ids'][0]) == df_dataset[params['dataset']['content_column']][0]
df_dataset[params['dataset']['content_column']][0]

2024-09-29 15:18:20.656701: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-09-29 15:18:20.671149: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-09-29 15:18:20.675965: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-09-29 15:18:20.688307: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


'def test_DFA(self):\n        transitions = {\n            \'a\': {\'1\': \'a\', \'0\': \'b\'},\n            \'b\': {\'1\': \'b\', \'0\': \'a\'}\n        }\n\n        final = [\'a\']\n        start = \'a\'\n\n        self.assertEqual(False, DFA(transitions, start, final, "000111100"))\n        self.assertEqual(True, DFA(transitions, start, final, "111000011"))\n\n        transitions1 = {\n            \'0\': {\'0\': \'1\', \'1\': \'0\'},\n            \'1\': {\'0\': \'2\', \'1\': \'0\'},\n            \'2\': {\'0\': \'2\', \'1\': \'3\'},\n            \'3\': {\'0\': \'3\', \'1\': \'3\'}\n        }\n\n        final1 = [\'0\', \'1\', \'2\']\n        start1 = \'0\'\n\n        self.assertEqual(False, DFA(transitions1, start1, final1, "0001111"))\n        self.assertEqual(True, DFA(transitions1, start1, final1, "01010101"))\n\n        transitions2 = {\n            \'0\': {\'a\': \'0\', \'b\': \'1\'},\n            \'1\': {\'a\': \'0\', \'b\': \'2\'},\n            \'2\': {\'a\': \'3\', \'b\': \'2

In [21]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

actual_cases = actual_logit(
    logit_vocab_sample_tensor = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    tokenized_prompt = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
    )
actual_cases

[('def', 0.00030555014382116497),
 ('test', 0.009959377348423004),
 ('_', 0.7111697196960449),
 ('D', 0.0002664475468918681),
 ('FA', 0.015310260467231274),
 ('(', 0.2075965702533722),
 ('self', 0.142356738448143),
 ('):', 0.7956698536872864),
 ('\n', 0.9903011322021484),
 ('     ', 0.2580970823764801),
 ('transitions', 0.00033252863795496523),
 ('=', 0.8424184322357178),
 ('{', 0.3088832497596741),
 ('\n', 0.7759588360786438),
 ('         ', 0.8008356690406799),
 ("'", 0.332145094871521),
 ('a', 0.2342398315668106),
 ("':", 0.9017483592033386),
 ("{'", 0.3621208071708679),
 ('1', 0.008214809000492096),
 ("':", 0.8090277314186096),
 ("'", 0.6437082290649414),
 ('a', 0.14299869537353516),
 ("',", 0.7275657057762146),
 ("'", 0.9586458802223206),
 ('0', 0.23419086635112762),
 ("':", 0.996057391166687),
 ("'", 0.9527722597122192),
 ('b', 0.6182431578636169),
 ("'},", 0.8787501454353333),
 ('\n', 0.9783669114112854),
 ('         ', 0.9917137622833252),
 ("'", 0.9982050657272339),
 ('b', 0.9

#### Processing all the Batches

In [22]:
def batching_logits(tokenizer,tf_input_ids,size=10000):
    max_logit_token_prompt = []
    min_logit_token_prompt = []
    actual_logit_token_prompt = []

    
    soft = torch.nn.Softmax( dim = 0 )                          #Flattening normalization
    
    for file in range( size ):
        out = np.load(params['callbacks_dir']+ '/'+ params['current_model'] + '_q_' + params['quantization'] +'/'+ f'logits_tensor[{file}]_batch[{file}].npy') #<sample,tokens,voc_tokens>
        out = out[0]  ##### #<tokens,voc_tokens>
        next_tokens_distribution = [ soft( torch.from_numpy(token) ) for token in out]  #Flattening normalization
        
        max_cases,min_cases = min_max_logits(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenizer_fn= tokenizer
            )

        actual_cases = actual_logit(
            logit_vocab_sample_tensor = next_tokens_distribution,
            tokenized_prompt = tf_input_ids[ file ],
            tokenizer_fn = tokenizer
            )
        
        max_logit_token_prompt.append( max_cases )
        min_logit_token_prompt.append( min_cases )
        actual_logit_token_prompt.append( actual_cases )
        
        logging.info(file)
    return max_logit_token_prompt,min_logit_token_prompt,actual_logit_token_prompt

In [23]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [24]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = batching_logits(
    tokenizer=tokenizer , tf_input_ids=input_ids_list, 
    size = len(df_dataset)
) #<---WARNING TIME Consuming

#### Saving results

In [25]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [26]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(55, 30)

In [27]:
dataframe_to_save.head(5)

,msg_id,line,column,end_line,end_column,code_smell,code,func_name,commit_id,repo,...,vocab_size,nloc,token_counts,n_identifiers,repository,year,input_ids,max_prob,min_prob,actual_prob
0,C0103,1,0,1,12,def test_DFA(self):,def test_DFA(self):\n transitions = {\n...,test_DFA,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,64,29,264,16,algorithms,outputs-22,"[1569, 2137, 29498, 29525, 3888, 29500, 1712, ...","[(#, 0.2665287256240845), (solution, 0.0588438...","[(–,, 3.8730735285597007e-10), (/***/, 1.16526...","[(def, 0.00030555014382116497), (test, 0.00995..."
1,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995..."
2,C0103,3,8,3,9,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995..."
3,C0103,4,8,4,9,"C = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,28,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995..."
4,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,30,11,117,11,algorithms,outputs-22,"[1569, 2137, 29498, 11329, 29498, 2933, 29498,...","[(#, 0.2665267884731293), (solution, 0.0588442...","[(–,, 3.8731712281858677e-10), (/***/, 1.16526...","[(def, 0.0003055557608604431), (test, 0.009959..."


In [28]:
create_folder(params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'])
dataframe_to_save.to_csv( params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'raw_logits.csv')

#### Loss Retrieval

In [29]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_loss = []
    for current_batch in range(size):
        out = np.load(params['callbacks_dir']+ '/'+ params['current_model'] +  '_q_' + params['quantization'] +'/' + f'_loss_batch[{current_batch}].npy') 
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [30]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [31]:
output_loss

[0.4558630585670471,
 0.6256284117698669,
 0.6256284117698669,
 0.6256284117698669,
 0.7624989151954651,
 0.7624989151954651,
 0.7624989151954651,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.6819478273391724,
 0.8713245987892151,
 0.25881484150886536,
 1.7146179676055908,
 1.5688334703445435,
 1.704850196838379,
 0.8083615899085999,
 1.0826220512390137,
 1.163515329360962,
 0.5581451654434204,
 0.5581451654434204,
 0.5581451654434204,
 0.5581451654434204,
 0.5581451654434204,
 0.5581451654434204,
 1.1955304145812988,
 1.1140497922897339,
 1.7491703033447266,
 1.8386893272399902,
 1.8386893272399902,
 1.63716459274292,
 1.63716459274292,
 2.856851100921631,
 0.8664145469665527,
 0.8664145469665527,
 0.5473245978355408,
 0.5473245978355408,
 0.9799413681030273,
 0.7896794080734253,
 0.8672385811805725,
 1.1682058572769165,
 1.4245681762695312

In [32]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,msg_id,line,column,end_line,end_column,code_smell,code,func_name,commit_id,repo,...,nloc,token_counts,n_identifiers,repository,year,input_ids,max_prob,min_prob,actual_prob,loss
0,C0103,1,0,1,12,def test_DFA(self):,def test_DFA(self):\n transitions = {\n...,test_DFA,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,29,264,16,algorithms,outputs-22,"[1569, 2137, 29498, 29525, 3888, 29500, 1712, ...","[(#, 0.2665287256240845), (solution, 0.0588438...","[(–,, 3.8730735285597007e-10), (/***/, 1.16526...","[(def, 0.00030555014382116497), (test, 0.00995...",0.455863
1,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995...",0.625628
2,C0103,3,8,3,9,"B = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995...",0.625628
3,C0103,4,8,4,9,"C = [2, 3, 3, 4]",def test_array_sum_combinations(self):\n ...,test_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,165,11,algorithms,outputs-22,"[1569, 2137, 29498, 2933, 29498, 2569, 29498, ...","[(#, 0.26652711629867554), (solution, 0.058844...","[(–,, 3.87315374217323e-10), (/***/, 1.1652591...","[(def, 0.00030555526609532535), (test, 0.00995...",0.625628
4,C0103,2,8,2,9,"A = [1, 2, 3, 3]",def test_unique_array_sum_combinations(self):\...,test_unique_array_sum_combinations,87eae9a25751a13d4ef48860fffa4df0998356eb,algorithms,...,11,117,11,algorithms,outputs-22,"[1569, 2137, 29498, 11329, 29498, 2933, 29498,...","[(#, 0.2665267884731293), (solution, 0.0588442...","[(–,, 3.8731712281858677e-10), (/***/, 1.16526...","[(def, 0.0003055557608604431), (test, 0.009959...",0.762499


In [33]:
## Saving CheckPoint 2
dataframe_to_save.to_csv( params['output_path'] + '/' + params['current_model'] + '_q_' + params['quantization'] + '/' + 'raw_logits.csv')

In [34]:
torch.cuda.empty_cache()
gc.collect()

154